# Lista 14 · Automação

O coletor do capítulo 14, peça por peça: primeiro ler a saída dos comandos
(texto, como no capítulo 9), depois conectar nos switches simulados, tratar as
falhas e montar o relatório.

A célula de preparo põe no ar os **switches simulados**, com a mesma interface da
biblioteca Netmiko: `ConnectHandler`, `send_command` e as exceções
`NetmikoTimeoutException` e `NetmikoAuthenticationException`. Usuário `noc`, senha
`marenet`. Os switches `10.0.1.20`, `10.0.2.20` e `10.0.5.20` respondem; o `10.0.3.20`
não responde e o `10.0.4.20` recusa a senha — de propósito.

---

**Como usar este caderno:** cada exercício tem duas células. Na primeira,
escreva a sua solução no lugar do `# TODO`. A segunda tem os testes —
rode-a e ela diz se a sua função está correta. Não altere a célula de teste.

Se um teste falhar, o Python mostra um `AssertionError` apontando a linha:
é aquele caso específico que a sua função ainda não atende.

**Comece pela célula abaixo.** Ela cria os arquivos de exemplo que os
exercícios desta lista leem. Sem ela, os testes falham com
`FileNotFoundError`.

In [ ]:
# simulador: equipamentos SSH da Maré Net — não precisa ler (faz o papel dos switches).
# Imita a interface da biblioteca Netmiko: ConnectHandler(...), send_command(...),
# disconnect() e as mesmas exceções. Com equipamentos de verdade, a única linha que
# muda é o import: from netmiko import ConnectHandler, ...
import time


class NetmikoTimeoutException(Exception):
    """O equipamento não respondeu a tempo."""


class NetmikoAuthenticationException(Exception):
    """Usuário ou senha recusados."""


_VERSAO = """Cisco IOS Software, C2960X Software (C2960X-UNIVERSALK9-M), Version {versao}, RELEASE SOFTWARE (fc3)
{nome} uptime is {uptime}
System image file is "flash:c2960x-universalk9-mz.{versao}.bin"
cisco WS-C2960X-48FPD-L (APM86XXX) processor with 524288K bytes of memory."""

_CABECALHO = "Interface              IP-Address      OK? Method Status                Protocol"

_EQUIPAMENTOS = {
    "10.0.1.20": {"nome": "SWITCH-CENTRO-01", "versao": "15.2(7)E7", "uptime": "3 weeks, 2 days, 4 hours",
                  "interfaces": [("Vlan10", "10.0.1.20", "up", "up"),
                                 ("GigabitEthernet1/0/1", "unassigned", "up", "up"),
                                 ("GigabitEthernet1/0/2", "unassigned", "up", "up"),
                                 ("GigabitEthernet1/0/3", "unassigned", "down", "down")]},
    "10.0.2.20": {"nome": "SWITCH-NORTE-02", "versao": "15.2(7)E7", "uptime": "12 days, 7 hours",
                  "interfaces": [("Vlan20", "10.0.2.20", "up", "up"),
                                 ("GigabitEthernet1/0/1", "unassigned", "up", "up"),
                                 ("GigabitEthernet1/0/2", "unassigned", "administratively down", "down"),
                                 ("GigabitEthernet1/0/3", "unassigned", "down", "down"),
                                 ("GigabitEthernet1/0/4", "unassigned", "down", "down")]},
    "10.0.3.20": {"nome": "SWITCH-SUL-03", "falha": "tempo"},
    "10.0.4.20": {"nome": "SWITCH-LESTE-04", "falha": "senha"},
    "10.0.5.20": {"nome": "SWITCH-OESTE-05", "versao": "15.2(4)E10", "uptime": "1 year, 5 weeks",
                  "interfaces": [("Vlan50", "10.0.5.20", "up", "up"),
                                 ("GigabitEthernet1/0/1", "unassigned", "up", "up"),
                                 ("GigabitEthernet1/0/2", "unassigned", "up", "up")]},
}
_SENHA = "marenet"


class _Conexao:
    def __init__(self, dados):
        self._dados = dados

    def send_command(self, comando):
        comando = " ".join(comando.split())
        if comando == "show version":
            return _VERSAO.format(**self._dados)
        if comando == "show ip interface brief":
            linhas = [_CABECALHO]
            for nome, ip, estado, protocolo in self._dados["interfaces"]:
                linhas.append(f"{nome:<23}{ip:<16}YES manual {estado:<22}{protocolo}")
            return "\n".join(linhas)
        if comando == "show clock":
            return "*14:03:17.123 BRT Mon Mar 2 2026"
        return "% Invalid input detected at '^' marker."

    def disconnect(self):
        pass

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.disconnect()


def ConnectHandler(device_type, host, username, password, timeout=5, **extras):
    """Abre uma "sessão SSH" com o equipamento simulado."""
    dados = _EQUIPAMENTOS.get(host)
    if dados is None or dados.get("falha") == "tempo":
        time.sleep(0.2)
        raise NetmikoTimeoutException(f"TCP connection to device failed: {host}")
    if dados.get("falha") == "senha" or password != _SENHA:
        raise NetmikoAuthenticationException(f"Authentication to device failed: {host}")
    return _Conexao(dados)

print("equipamentos simulados prontos (senha do usuário noc: marenet)")

### Exercício 01

Escreva `versao_de(saida)`, que recebe a saída do `show version` e devolve a versão
do sistema: o texto entre `Version ` e a vírgula seguinte.

```
Cisco IOS Software, C2960X Software (C2960X-UNIVERSALK9-M), Version 15.2(7)E7, RELEASE SOFTWARE (fc3)
SWITCH-CENTRO-01 uptime is 3 weeks, 2 days, 4 hours
System image file is "flash:c2960x-universalk9-mz.15.2(7)E7.bin"
```

Para essa saída, a resposta é `"15.2(7)E7"`.

In [ ]:
def versao_de(saida):
    """A versão do sistema, tirada da saída do show version."""
    # TODO: ache a linha com "Version "; partition("Version ") e depois partition(",")
    pass

In [ ]:
# Célula de teste — Exercício 01
assert versao_de('Cisco IOS Software, C2960X Software (C2960X-UNIVERSALK9-M), Version 15.2(7)E7, RELEASE SOFTWARE (fc3)\nSWITCH-CENTRO-01 uptime is 3 weeks, 2 days, 4 hours\nSystem image file is "flash:c2960x-universalk9-mz.15.2(7)E7.bin"') == '15.2(7)E7', versao_de('Cisco IOS Software, C2960X Software (C2960X-UNIVERSALK9-M), Version 15.2(7)E7, RELEASE SOFTWARE (fc3)\nSWITCH-CENTRO-01 uptime is 3 weeks, 2 days, 4 hours\nSystem image file is "flash:c2960x-universalk9-mz.15.2(7)E7.bin"')
assert versao_de('saida sem a palavra') is None, versao_de('saida sem a palavra')
print("Exercício 01: todos os testes passaram!")

### Exercício 02

Escreva `uptime_de(saida)`, que devolve o texto depois de `"uptime is "` na saída
do `show version` (a mesma do exercício 01), ou `None` se não houver.

In [ ]:
def uptime_de(saida):
    """Há quanto tempo o equipamento está ligado, como texto."""
    # TODO: mesma ideia do exercício 01, com "uptime is "
    pass

In [ ]:
# Célula de teste — Exercício 02
assert uptime_de('Cisco IOS Software, C2960X Software (C2960X-UNIVERSALK9-M), Version 15.2(7)E7, RELEASE SOFTWARE (fc3)\nSWITCH-CENTRO-01 uptime is 3 weeks, 2 days, 4 hours\nSystem image file is "flash:c2960x-universalk9-mz.15.2(7)E7.bin"') == '3 weeks, 2 days, 4 hours', uptime_de('Cisco IOS Software, C2960X Software (C2960X-UNIVERSALK9-M), Version 15.2(7)E7, RELEASE SOFTWARE (fc3)\nSWITCH-CENTRO-01 uptime is 3 weeks, 2 days, 4 hours\nSystem image file is "flash:c2960x-universalk9-mz.15.2(7)E7.bin"')
assert uptime_de('') is None, uptime_de('')
print("Exercício 02: todos os testes passaram!")

### Exercício 03

Escreva `estados(saida)`, que recebe a saída do `show ip interface brief` e devolve
um dicionário **interface → estado**. Cuidado: o estado pode ter espaço
(`administratively down`).

```
Interface              IP-Address      OK? Method Status                Protocol
Vlan20                 10.0.2.20       YES manual up                    up
GigabitEthernet1/0/1   unassigned      YES manual up                    up
GigabitEthernet1/0/2   unassigned      YES manual administratively down down
GigabitEthernet1/0/3   unassigned      YES manual down                  down
```

In [ ]:
def estados(saida):
    """Dicionário interface -> estado, do show ip interface brief."""
    # TODO: pule o cabeçalho (splitlines()[1:]); split(maxsplit=4) e, no resto,
    #       rpartition(" ") separa o protocolo do estado; strip() no estado
    pass

In [ ]:
# Célula de teste — Exercício 03
assert estados('Interface              IP-Address      OK? Method Status                Protocol\nVlan20                 10.0.2.20       YES manual up                    up\nGigabitEthernet1/0/1   unassigned      YES manual up                    up\nGigabitEthernet1/0/2   unassigned      YES manual administratively down down\nGigabitEthernet1/0/3   unassigned      YES manual down                  down') == {'Vlan20': 'up', 'GigabitEthernet1/0/1': 'up', 'GigabitEthernet1/0/2': 'administratively down', 'GigabitEthernet1/0/3': 'down'}, estados('Interface              IP-Address      OK? Method Status                Protocol\nVlan20                 10.0.2.20       YES manual up                    up\nGigabitEthernet1/0/1   unassigned      YES manual up                    up\nGigabitEthernet1/0/2   unassigned      YES manual administratively down down\nGigabitEthernet1/0/3   unassigned      YES manual down                  down')
print("Exercício 03: todos os testes passaram!")

### Exercício 04

Escreva `interfaces_paradas(saida)`, que devolve a lista das interfaces **cujo
estado não é `up`** (na ordem da saída). `estados` já está no esqueleto.

In [ ]:
def estados(saida):
    """JÁ ESCRITA."""
    resultado = {}
    for linha in saida.splitlines()[1:]:
        interface, ip, ok, metodo, resto = linha.split(maxsplit=4)
        estado, _, protocolo = resto.rpartition(" ")
        resultado[interface] = estado.strip()
    return resultado


def interfaces_paradas(saida):
    """Interfaces com estado diferente de up."""
    # TODO: o filtro sobre estados(saida).items()
    pass

In [ ]:
# Célula de teste — Exercício 04
assert interfaces_paradas('Interface              IP-Address      OK? Method Status                Protocol\nVlan20                 10.0.2.20       YES manual up                    up\nGigabitEthernet1/0/1   unassigned      YES manual up                    up\nGigabitEthernet1/0/2   unassigned      YES manual administratively down down\nGigabitEthernet1/0/3   unassigned      YES manual down                  down') == ['GigabitEthernet1/0/2', 'GigabitEthernet1/0/3'], interfaces_paradas('Interface              IP-Address      OK? Method Status                Protocol\nVlan20                 10.0.2.20       YES manual up                    up\nGigabitEthernet1/0/1   unassigned      YES manual up                    up\nGigabitEthernet1/0/2   unassigned      YES manual administratively down down\nGigabitEthernet1/0/3   unassigned      YES manual down                  down')
print("Exercício 04: todos os testes passaram!")

### Exercício 05

Escreva `roda_comando(host, comando)`, que conecta no equipamento (usuário `noc`,
senha `marenet`, `device_type="cisco_ios"`) **com `with`**, manda o comando e devolve
a saída. Os switches simulados estão em `10.0.1.20`, `10.0.2.20` e `10.0.5.20`.

In [ ]:
def roda_comando(host, comando):
    """Saída de um comando num equipamento."""
    # TODO: with ConnectHandler(device_type=..., host=host, username=..., password=...) as conexao:
    #           return conexao.send_command(comando)
    pass

In [ ]:
# Célula de teste — Exercício 05
assert roda_comando('10.0.1.20', 'show clock') == '*14:03:17.123 BRT Mon Mar 2 2026', roda_comando('10.0.1.20', 'show clock')
assert roda_comando('10.0.5.20', 'show hora') == "% Invalid input detected at '^' marker.", roda_comando('10.0.5.20', 'show hora')
print("Exercício 05: todos os testes passaram!")

### Exercício 06

Escreva `hostname_de(host)`, que roda `show version` no equipamento e devolve o
nome dele — a primeira palavra da linha que contém `"uptime is"`.

In [ ]:
def hostname_de(host):
    """Nome do equipamento, tirado do show version."""
    # TODO: conecte, rode o show version, ache a linha com "uptime is" e
    #       pegue o primeiro campo do split()
    pass

In [ ]:
# Célula de teste — Exercício 06
assert hostname_de('10.0.1.20') == 'SWITCH-CENTRO-01', hostname_de('10.0.1.20')
assert hostname_de('10.0.5.20') == 'SWITCH-OESTE-05', hostname_de('10.0.5.20')
print("Exercício 06: todos os testes passaram!")

### Exercício 07

Escreva `tenta_conectar(host)`, que devolve `"ok"` se conseguir conectar,
`"tempo esgotado"` se der `NetmikoTimeoutException` e `"senha recusada"` se der
`NetmikoAuthenticationException`. Os switches `10.0.3.20` e `10.0.4.20` falham de
propósito.

In [ ]:
def tenta_conectar(host):
    """"ok", "tempo esgotado" ou "senha recusada"."""
    # TODO: um try com o with ConnectHandler(...); um except para cada exceção
    pass

In [ ]:
# Célula de teste — Exercício 07
assert tenta_conectar('10.0.1.20') == 'ok', tenta_conectar('10.0.1.20')
assert tenta_conectar('10.0.3.20') == 'tempo esgotado', tenta_conectar('10.0.3.20')
assert tenta_conectar('10.0.4.20') == 'senha recusada', tenta_conectar('10.0.4.20')
print("Exercício 07: todos os testes passaram!")

### Exercício 08

Escreva `coleta(item)`, que recebe um dicionário `{"nome": ..., "host": ...}`,
conecta, lê a versão com `show version` e devolve um `Resultado` (já escrito) — com
`ok=True` e a versão, ou com `ok=False` e o motivo (`"tempo esgotado"` ou
`"senha recusada"`). A função **nunca** deixa uma exceção escapar.

In [ ]:
from dataclasses import dataclass


@dataclass
class Resultado:
    """O resultado da coleta de um equipamento. JÁ ESCRITA."""
    nome: str
    ok: bool
    versao: str = ""
    erro: str = ""


def versao_de(saida):
    """JÁ ESCRITA."""
    for linha in saida.splitlines():
        if "Version " in linha:
            return linha.partition("Version ")[2].partition(",")[0]
    return None


def coleta(item):
    """Resultado da coleta de um item {"nome": ..., "host": ...}; nunca levanta erro."""
    # TODO: try com o with ConnectHandler(...); versao_de(send_command("show version"));
    #       return Resultado(item["nome"], True, versao); um except por falha
    pass

In [ ]:
# Célula de teste — Exercício 08
assert coleta({'nome': 'SW-CENTRO', 'host': '10.0.1.20'}) == Resultado(nome='SW-CENTRO', ok=True, versao='15.2(7)E7', erro=''), coleta({'nome': 'SW-CENTRO', 'host': '10.0.1.20'})
assert coleta({'nome': 'SW-SUL', 'host': '10.0.3.20'}) == Resultado(nome='SW-SUL', ok=False, versao='', erro='tempo esgotado'), coleta({'nome': 'SW-SUL', 'host': '10.0.3.20'})
assert coleta({'nome': 'SW-LESTE', 'host': '10.0.4.20'}) == Resultado(nome='SW-LESTE', ok=False, versao='', erro='senha recusada'), coleta({'nome': 'SW-LESTE', 'host': '10.0.4.20'})
print("Exercício 08: todos os testes passaram!")

### Exercício 09

Escreva `coleta_todos(inventario)`, que devolve a lista de `Resultado`, um por
item do inventário, na ordem. `coleta` já está no esqueleto.

In [ ]:
from dataclasses import dataclass


@dataclass
class Resultado:
    """O resultado da coleta de um equipamento. JÁ ESCRITA."""
    nome: str
    ok: bool
    versao: str = ""
    erro: str = ""


def versao_de(saida):
    """JÁ ESCRITA."""
    for linha in saida.splitlines():
        if "Version " in linha:
            return linha.partition("Version ")[2].partition(",")[0]
    return None


def coleta(item):
    """Resultado da coleta de um item {"nome": ..., "host": ...}; nunca levanta erro. JÁ ESCRITA."""
    try:
        with ConnectHandler(device_type="cisco_ios", host=item["host"],
                            username="noc", password="marenet") as conexao:
            versao = versao_de(conexao.send_command("show version"))
        return Resultado(item["nome"], True, versao)
    except NetmikoTimeoutException:
        return Resultado(item["nome"], False, erro="tempo esgotado")
    except NetmikoAuthenticationException:
        return Resultado(item["nome"], False, erro="senha recusada")


def coleta_todos(inventario):
    """Lista de Resultado, um por item do inventário."""
    # TODO: o laço de sempre, com append(coleta(item))
    pass

In [ ]:
# Célula de teste — Exercício 09
assert coleta_todos([{'nome': 'SW-CENTRO', 'host': '10.0.1.20'}, {'nome': 'SW-SUL', 'host': '10.0.3.20'}, {'nome': 'SW-LESTE', 'host': '10.0.4.20'}, {'nome': 'SW-OESTE', 'host': '10.0.5.20'}]) == [Resultado(nome='SW-CENTRO', ok=True, versao='15.2(7)E7', erro=''), Resultado(nome='SW-SUL', ok=False, versao='', erro='tempo esgotado'), Resultado(nome='SW-LESTE', ok=False, versao='', erro='senha recusada'), Resultado(nome='SW-OESTE', ok=True, versao='15.2(4)E10', erro='')], coleta_todos([{'nome': 'SW-CENTRO', 'host': '10.0.1.20'}, {'nome': 'SW-SUL', 'host': '10.0.3.20'}, {'nome': 'SW-LESTE', 'host': '10.0.4.20'}, {'nome': 'SW-OESTE', 'host': '10.0.5.20'}])
assert coleta_todos([]) == [], coleta_todos([])
print("Exercício 09: todos os testes passaram!")

### Exercício 10

Escreva `relatorio(resultados)`, que recebe uma lista de `Resultado` e devolve um
texto: primeiro uma linha por equipamento que deu certo (`nome` em 12 colunas e a
versão), depois uma linha por falha (`nome` em 12 colunas, `FALHOU: ` e o motivo), e
por fim `"3 de 5 coletados"` com os números certos.

```
SW-CENTRO   15.2(7)E7
SW-OESTE    15.2(4)E10
SW-SUL      FALHOU: tempo esgotado
SW-LESTE    FALHOU: senha recusada
2 de 4 coletados
```

In [ ]:
from dataclasses import dataclass


@dataclass
class Resultado:
    """O resultado da coleta de um equipamento. JÁ ESCRITA."""
    nome: str
    ok: bool
    versao: str = ""
    erro: str = ""


def relatorio(resultados):
    """Relatório da coleta: sucessos, falhas e o total."""
    # TODO: um laço para os ok, outro para as falhas, a linha final; "\n".join
    pass

In [ ]:
# Célula de teste — Exercício 10
assert relatorio([Resultado('SW-CENTRO', True, '15.2(7)E7'), Resultado('SW-SUL', False, erro='tempo esgotado'), Resultado('SW-LESTE', False, erro='senha recusada'), Resultado('SW-OESTE', True, '15.2(4)E10')]) == 'SW-CENTRO   15.2(7)E7\nSW-OESTE    15.2(4)E10\nSW-SUL      FALHOU: tempo esgotado\nSW-LESTE    FALHOU: senha recusada\n2 de 4 coletados', relatorio([Resultado('SW-CENTRO', True, '15.2(7)E7'), Resultado('SW-SUL', False, erro='tempo esgotado'), Resultado('SW-LESTE', False, erro='senha recusada'), Resultado('SW-OESTE', True, '15.2(4)E10')])
print("Exercício 10: todos os testes passaram!")